# `transform_geoprocessing.ipynb` - Geoprocesamiento espacial (Spatial Join)

Requisito: *"GeoPandas: Geoprocesamiento vectorial, Spatial Join (gpd.sjoin), manejo de CRS y GeoJSON."*

Este módulo convierte los puntos de recarga (lat/lon) en un `GeoDataFrame` de puntos y los cruza espacialmente con los polígonos de distritos mediante `gpd.sjoin(..., predicate="within")`: cada punto de recarga queda asignado al distrito cuyo polígono lo contiene. Los puntos que no caen dentro de ninguno de los 11 distritos (p. ej. en municipios colindantes dentro del radio de 20 km, o en la Isla de la Cartuja) se desvían a una tabla de cuarentena - el mismo principio de calidad de datos que `tb_cuarentena_geodatos` en el proyecto GeoStat, aplicado aquí a nivel espacial en vez de tabular.

> Depende de `config.ipynb` y `logging_config.ipynb` (usa `logger`).

In [ ]:
import logging

import geopandas as gpd
import pandas as pd

## 1. Puntos de recarga -> GeoDataFrame

In [ ]:
def estaciones_a_geodataframe(estaciones: list) -> gpd.GeoDataFrame:
    """Convierte la lista de dicts {nombre, lat, lon, potencia_kw} en un GeoDataFrame de puntos (EPSG:4326)."""
    df = pd.DataFrame(estaciones)
    gdf = gpd.GeoDataFrame(
        df,
        geometry=gpd.points_from_xy(df["lon"], df["lat"]),
        crs=CRS_GEOGRAFICO,
    )
    return gdf

## 2. Spatial Join - asignar cada punto a su distrito

Requisito de cuarentena espacial: los puntos que no caen dentro de ningún distrito se separan (no se descartan silenciosamente) para que quede constancia de cuántos y cuáles fueron excluidos del cálculo del IOI.

In [ ]:
def cruzar_estaciones_con_distritos(gdf_estaciones: gpd.GeoDataFrame, gdf_distritos: gpd.GeoDataFrame):
    """
    Spatial join: asigna cada punto de recarga al distrito que lo
    contiene (predicate="within"). Devuelve (df_validos, df_cuarentena).
    """
    logger.info("Cruzando puntos de recarga con poligonos de distrito (gpd.sjoin, predicate=within)...")

    join = gpd.sjoin(
        gdf_estaciones,
        gdf_distritos[["nombre_distrito", "geometry"]],
        how="left",
        predicate="within",
    )

    mask_validos = join["nombre_distrito"].notna()
    df_validos = join[mask_validos][["nombre", "lat", "lon", "potencia_kw", "nombre_distrito"]].copy()
    df_cuarentena = join[~mask_validos][["nombre", "lat", "lon", "potencia_kw"]].copy()
    df_cuarentena["motivo_rechazo"] = "Punto fuera de los 11 distritos oficiales de Sevilla"

    logger.info(
        f"Spatial join completado. Validos: {len(df_validos)} | "
        f"Cuarentena (fuera de distrito): {len(df_cuarentena)}"
    )
    return df_validos, df_cuarentena

## 3. Agregación por distrito

Suma de potencia instalada y recuento de estaciones por distrito, tal como exige la fórmula de Densidad Energética. Los distritos sin ningún punto de recarga válido se completan con 0 (no desaparecen del resultado).

In [ ]:
def agregar_potencia_por_distrito(df_validos: pd.DataFrame, nombres_distritos: list) -> pd.DataFrame:
    """Suma potencia_kw y cuenta estaciones por distrito; rellena con 0 los distritos sin puntos."""
    agregado = df_validos.groupby("nombre_distrito", as_index=False).agg(
        potencia_total_kw=("potencia_kw", "sum"),
        num_estaciones=("potencia_kw", "count"),
    )
    # Asegurar que los 11 distritos aparecen, aunque no tengan puntos de recarga
    base = pd.DataFrame({"nombre_distrito": nombres_distritos})
    agregado = base.merge(agregado, on="nombre_distrito", how="left").fillna(
        {"potencia_total_kw": 0.0, "num_estaciones": 0}
    )
    agregado["num_estaciones"] = agregado["num_estaciones"].astype(int)
    return agregado

## Prueba rápida

Con el dataset de respaldo (`ESTACIONES_FALLBACK`, de `config.ipynb`) y los distritos reales, sin depender de la API.

In [ ]:
gdf_estaciones_prueba = estaciones_a_geodataframe(ESTACIONES_FALLBACK)
gdf_distritos_prueba2 = extraer_distritos()

df_validos_prueba, df_cuarentena_prueba = cruzar_estaciones_con_distritos(gdf_estaciones_prueba, gdf_distritos_prueba2)
print(f"Validos: {len(df_validos_prueba)} | Cuarentena: {len(df_cuarentena_prueba)}")
if len(df_cuarentena_prueba):
    print("\nEn cuarentena (fuera de los 11 distritos):")
    print(df_cuarentena_prueba[["nombre", "motivo_rechazo"]].to_string(index=False))

df_agregado_prueba = agregar_potencia_por_distrito(df_validos_prueba, sorted(POBLACION_DISTRITOS.keys()))
print("\nPotencia agregada por distrito:")
print(df_agregado_prueba.sort_values("potencia_total_kw", ascending=False).to_string(index=False))

---
✅ **Geoprocesamiento verificado**: el spatial join asigna correctamente los puntos a sus distritos y desvía a cuarentena los que quedan fuera (p. ej. municipios colindantes o la Isla de la Cartuja).